# Stage 4: KAN-Phase Implementation
**Project:** KAN-Driven Phase-Spectrum Analysis for Deepfake Detection

In [ ]:
!pip install pykan kagglehub -q

In [ ]:
import numpy as np, pandas as pd, os, glob, json, time
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from kan import KAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, classification_report, confusion_matrix, roc_curve
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.notebook import tqdm
import kagglehub

%matplotlib inline
plt.rcParams['figure.dpi']=120; sns.set_style('whitegrid')

INPUT_DIR = kagglehub.dataset_download('awsaf49/artifact-dataset')
OUTPUT_DIR = '/kaggle/working' if os.path.isdir('/kaggle/working') else './output'
CACHE_DIR = os.path.join(OUTPUT_DIR, 'phase_cache')
MODEL_DIR = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:{DEVICE} Dataset:{INPUT_DIR}')
CONFIG = {'n_pca':128,'width':[128,64,32,1],'grid':5,'k':3,'lr':1e-3,'wd':1e-4,'bs':64,'epochs':50,'patience':10}

In [ ]:
# Cell 2: Load & Prep
cache_files = sorted(glob.glob(os.path.join(CACHE_DIR,'phase_maps_*.npy')))
phase_results = np.load(cache_files[-1], allow_pickle=True).item()
meta_files = glob.glob(os.path.join(INPUT_DIR,'**','metadata.csv'), recursive=True)
mdf = pd.concat([pd.read_csv(mf).assign(generator=os.path.basename(os.path.dirname(mf))) for mf in meta_files], ignore_index=True) if meta_files else None
X_l,y_l=[],[]
for g,maps in phase_results.items():
    if mdf is not None: gdf=mdf[mdf['generator']==g]; ir=len(gdf)>0 and gdf['target'].mode().iloc[0]==0
    else: ir='real' in g.lower()
    for m in maps: X_l.append(m.flatten()); y_l.append(0 if ir else 1)
X_raw=np.array(X_l,dtype=np.float32); y=np.array(y_l,dtype=np.int64)
sc=StandardScaler(); pc=PCA(n_components=CONFIG['n_pca'],random_state=42)
X_pca=pc.fit_transform(sc.fit_transform(X_raw)).astype(np.float32)
print(f'PCA:{X_pca.shape} var:{pc.explained_variance_ratio_.sum():.3f}')
X_tv,X_te,y_tv,y_te=train_test_split(X_pca,y,test_size=0.2,stratify=y,random_state=42)
X_tr,X_va,y_tr,y_va=train_test_split(X_tv,y_tv,test_size=0.15,stratify=y_tv,random_state=42)
print(f'Train:{len(X_tr)} Val:{len(X_va)} Test:{len(X_te)}')
trl=DataLoader(TensorDataset(torch.FloatTensor(X_tr),torch.LongTensor(y_tr)),batch_size=64,shuffle=True)
val=DataLoader(TensorDataset(torch.FloatTensor(X_va),torch.LongTensor(y_va)),batch_size=64)
tel=DataLoader(TensorDataset(torch.FloatTensor(X_te),torch.LongTensor(y_te)),batch_size=64)

In [ ]:
# Cell 3: Train KAN
kan=KAN(width=CONFIG['width'],grid=CONFIG['grid'],k=CONFIG['k'],device=str(DEVICE))
np_k=sum(p.numel() for p in kan.parameters()); print(f'KAN:{CONFIG["width"]} grid={CONFIG["grid"]} params={np_k:,}')
crit=nn.BCEWithLogitsLoss(); opt=optim.Adam(kan.parameters(),lr=CONFIG['lr'],weight_decay=CONFIG['wd'])
sched=optim.lr_scheduler.ReduceLROnPlateau(opt,mode='max',factor=0.5,patience=5)
hist={'tl':[],'vl':[],'ta':[],'va':[]}
best,pat=0.0,0
for ep in tqdm(range(CONFIG['epochs']),desc='KAN'):
    kan.train(); tl,tp,tt=0,[],[]
    for xb,yb in trl:
        xb,yb=xb.to(DEVICE),yb.float().to(DEVICE); opt.zero_grad()
        lo=kan(xb).squeeze(-1); l=crit(lo,yb); l.backward(); opt.step()
        tl+=l.item()*len(yb); tp.append(torch.sigmoid(lo).detach().cpu().numpy()); tt.append(yb.cpu().numpy())
    tp,tt=np.concatenate(tp),np.concatenate(tt); ta=roc_auc_score(tt,tp) if len(np.unique(tt))>1 else 0
    kan.eval(); vl,vp,vt=0,[],[]
    with torch.no_grad():
        for xb,yb in val:
            xb,yb=xb.to(DEVICE),yb.float().to(DEVICE); lo=kan(xb).squeeze(-1)
            vl+=crit(lo,yb).item()*len(yb); vp.append(torch.sigmoid(lo).cpu().numpy()); vt.append(yb.cpu().numpy())
    vp,vt=np.concatenate(vp),np.concatenate(vt); va2=roc_auc_score(vt,vp) if len(np.unique(vt))>1 else 0
    sched.step(va2)
    hist['tl'].append(tl/len(trl.dataset));hist['vl'].append(vl/len(val.dataset));hist['ta'].append(ta);hist['va'].append(va2)
    if va2>best: best=va2;pat=0;torch.save(kan.state_dict(),os.path.join(MODEL_DIR,'kan_best.pth'))
    else:
        pat+=1
        if pat>=CONFIG['patience']: print(f'Early stop ep {ep+1}'); break
kan.load_state_dict(torch.load(os.path.join(MODEL_DIR,'kan_best.pth'),weights_only=True))

In [ ]:
# Cell 4: Eval & A2
kan.eval(); ap,at=[],[]
with torch.no_grad():
    for xb,yb in tel: ap.append(torch.sigmoid(kan(xb.to(DEVICE)).squeeze(-1)).cpu().numpy()); at.append(yb.numpy())
kp,kt=np.concatenate(ap),np.concatenate(at)
kacc=accuracy_score(kt,(kp>0.5).astype(int)); kauc=roc_auc_score(kt,kp)
print(f'KAN-Phase: Acc={kacc:.4f} AUC={kauc:.4f} Params={np_k:,}')
print(classification_report(kt,(kp>0.5).astype(int),target_names=['Real','Fake']))

fig,(a1,a2)=plt.subplots(1,2,figsize=(12,5))
ep=range(1,len(hist['tl'])+1)
a1.plot(ep,hist['tl'],label='Train');a1.plot(ep,hist['vl'],label='Val');a1.set_title('Loss');a1.legend()
a2.plot(ep,hist['ta'],label='Train');a2.plot(ep,hist['va'],label='Val');a2.set_title('AUC');a2.legend()
plt.suptitle('KAN-Phase Training',fontweight='bold');plt.tight_layout();plt.show()

fig,(a1,a2)=plt.subplots(1,2,figsize=(12,5))
fpr,tpr,_=roc_curve(kt,kp);a1.plot(fpr,tpr,lw=2,label=f'AUC={kauc:.3f}');a1.plot([0,1],[0,1],'k--',alpha=0.3);a1.legend();a1.set_title('ROC')
cm=confusion_matrix(kt,(kp>0.5).astype(int))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',ax=a2,xticklabels=['Real','Fake'],yticklabels=['Real','Fake']);a2.set_title('Confusion')
plt.tight_layout();plt.show()

# A2 comparison
b3p=os.path.join(MODEL_DIR,'results_b3.json')
if os.path.exists(b3p):
    with open(b3p) as f: b3=json.load(f)
    print(f'\n{"="*50}\nABLATION A2: KAN vs MLP\n{"="*50}')
    comp=pd.DataFrame([{'Model':'KAN-Phase','AUC':kauc,'Acc':kacc,'Params':np_k},
        {'Model':b3['model'],'AUC':b3['test_auc'],'Acc':b3['test_accuracy'],'Params':b3['n_parameters']}])
    print(comp.to_string(index=False))
    comp.to_csv(os.path.join(MODEL_DIR,'ablation_a2.csv'),index=False)

with open(os.path.join(MODEL_DIR,'results_kan.json'),'w') as f:
    json.dump({'model':'KAN-Phase','test_accuracy':float(kacc),'test_auc':float(kauc),'n_parameters':np_k,
               'kan_width':CONFIG['width'],'kan_grid_size':CONFIG['grid']},f,indent=2)
print('KAN results saved.')